In [41]:
import duckdb
conn = duckdb.connect(r"C:\Users\eddiec11us\dev_apps\customer-matching-app\src\data\db.duckdb")
# conn.close()

ideal workflow
user opens the review queue and says 

"these are siblings, make a parent"
or
"these are siblings, add to existing parent"
or
"these do not belong to this parent, don't suggest it again"

and

"these vendor customers go to this erp account"
or
"these erp accounts go to this parent"

then:
someone comes in and says no wait, this one does not go to that parent, delete that relationship

finally:
add these relationships to data sets
store this data
export to excel for the normal people: master sheet: left cust, right cust, relationship type (?)


In [87]:
"""
candidates might have parents, if they do that is the suggested parent(s)
"""

candidates = f"""
WITH base AS (
SELECT
  vc.vendor_customer_id,
  vc.vendor_name,
  vc.normalized_vendor_customer_name,
  vc.first3_token,
  vc.billing_state,
  vc.billing_zip,
  p.parent_account_name
FROM vendor_customers vc

LEFT JOIN vendor_customer_to_parent_account_map pid ON 
  vc.vendor_customer_id = pid.vendor_customer_id
LEFT JOIN parent_accounts p ON
  pid.parent_account_id = p.parent_account_id

WHERE 
  vc.first3_token IS NOT NULL
  AND vc.billing_state IS NOT NULL
  AND vc.billing_zip IS NOT NULL
), 

counted AS (
SELECT 
  *,
  COUNT(*) OVER (
    PARTITION BY base.first3_token, base.billing_state, base.billing_zip
  ) AS sibling_count
FROM base
), 

filtered as (
  SELECT * FROM counted
WHERE sibling_count > 1

), 

ranked AS (
SELECT 
  *,
  DENSE_RANK() OVER (
  PARTITION BY filtered.first3_token, filtered.billing_state, filtered.billing_zip
  ) AS sibling_group
FROM filtered
)
SELECT 
  sibling_group,
  vendor_customer_id,
  parent_account_name,
  vendor_name,
  normalized_vendor_customer_name,
  first3_token,
  billing_state,
  billing_zip
FROM ranked
"""

candidates_df = conn.sql(query=candidates).df()
candidates_df

,sibling_group,vendor_customer_id,parent_account_name,vendor_name,normalized_vendor_customer_name,first3_token,billing_state,billing_zip
0,1,1,None,almo,eleven hospitality,ele,PA,17543
1,1,2,None,td synnex,eleven hospitality inc,ele,PA,17543
2,1,3,None,bluestar,eleven hospitality incorporated pa,ele,PA,17543
